In [2]:
"""
Generate comprehensive metadata files for CER and Comstock datasets
"""

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

root = Path(os.getcwd()).resolve().parents[0]
sys.path.append(str(root))

def analyze_timeseries_data(csv_path):
    """Analyze time series data and extract statistics"""
    df = pd.read_csv(csv_path)
    
    stats = {
        'num_samples': int(len(df)),  # Convert to int
        'num_features': int(len(df.columns)),  # Convert to int
        'feature_names': list(df.columns),
        'data_shape': f"{df.shape[0]} samples × {df.shape[1]} features",
        'memory_usage_mb': float(df.memory_usage(deep=True).sum() / (1024**2)),  # Convert to float
        'missing_values': int(df.isnull().sum().sum()),  # Convert to int
        'missing_percentage': float((df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100)  # Convert to float
    }
    
    # Numerical statistics for consumption data
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        stats['numerical_statistics'] = {
            'mean': float(df[numeric_cols].mean().mean()),
            'std': float(df[numeric_cols].std().mean()),
            'min': float(df[numeric_cols].min().min()),
            'max': float(df[numeric_cols].max().max()),
            'median': float(df[numeric_cols].median().median())
        }
    
    return stats

def analyze_labels(csv_path):
    """Analyze label files for classification tasks"""
    df = pd.read_csv(csv_path)
    
    label_stats = {
        'num_samples': int(len(df)),  # Convert to int
        'num_features': int(len(df.columns)),  # Convert to int
        'columns': list(df.columns),
        'label_distribution': {}
    }
    
    # Check for binary labels
    if 'label' in df.columns or any('label' in col.lower() for col in df.columns):
        label_col = [col for col in df.columns if 'label' in col.lower()][0]
        unique_labels = df[label_col].value_counts()
        label_stats['label_distribution'] = {
            str(k): int(v) for k, v in unique_labels.items()  # Convert values to int
        }
        label_stats['class_balance'] = {
            'positive_ratio': float(unique_labels.get(1, 0) / len(df)),  # Convert to float
            'negative_ratio': float(unique_labels.get(0, 0) / len(df))  # Convert to float
        }
    
    return label_stats

def generate_cer_metadata():
    """Generate metadata for CER (Irish residential) dataset"""
    
    data_path = root / 'data'
    
    metadata = {
        'dataset_name': 'CER (Commission for Energy Regulation) - Irish Residential',
        'dataset_type': 'Smart Meter Consumption Data',
        'source': 'CER Smart Metering Project',
        'country': 'Ireland',
        'sector': 'Residential',
        'description': 'Smart meter electricity consumption data from residential customers in Ireland',
        'temporal_resolution': '30 minutes',
        'date_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'reference': 'Commission for Energy Regulation (CER)',
        'license': 'Open Data',
        'use_cases': [
            'Appliance detection',
            'Load forecasting',
            'Energy consumption analysis',
            'Non-intrusive load monitoring (NILM)'
        ]
    }
    
    # Analyze input data
    input_file = data_path / 'Inputs' / 'x_residential_25728.csv'
    if input_file.exists():
        print(f"Analyzing {input_file}...")
        input_stats = analyze_timeseries_data(input_file)
        metadata['input_data'] = {
            'file': 'Inputs/x_residential_25728.csv',
            'description': 'Time series of electricity consumption readings',
            **input_stats
        }
    
    # Analyze exogenous data
    exog_file = data_path / 'ExogeneData' / 'extra_25728.csv'
    if exog_file.exists():
        print(f"Analyzing {exog_file}...")
        exog_stats = analyze_timeseries_data(exog_file)
        metadata['exogenous_data'] = {
            'file': 'ExogeneData/extra_25728.csv',
            'description': 'Temporal features (hours, days encoded in sin/cos)',
            **exog_stats
        }
    
    # Analyze label files
    labels_dir = data_path / 'Labels'
    if labels_dir.exists():
        label_files = list(labels_dir.glob('*.csv'))
        metadata['classification_tasks'] = {
            'num_appliances': len(label_files),
            'appliances': []
        }
        
        for label_file in sorted(label_files):
            print(f"Analyzing {label_file.name}...")
            appliance_name = label_file.stem.replace('_case', '').replace('_', ' ').title()
            label_stats = analyze_labels(label_file)
            
            metadata['classification_tasks']['appliances'].append({
                'name': appliance_name,
                'file': f'Labels/{label_file.name}',
                'description': f'Binary classification for {appliance_name} detection',
                **label_stats
            })
    
    # Save metadata
    output_file = data_path / 'CER_metadata.json'
    with open(output_file, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"\n✓ CER metadata saved to {output_file}")
    return metadata

def generate_comstock_metadata():
    """Generate metadata for ComStock (US commercial buildings) dataset"""
    
    data_path = root / 'data' / 'Comstock'
    comstock_15min_path = root / 'data' / 'Comstock_15min'
    
    metadata = {
        'dataset_name': 'ComStock - US Commercial Building Stock',
        'dataset_type': 'Synthetic Building Energy Consumption Data',
        'source': 'NREL ComStock',
        'country': 'United States',
        'sector': 'Commercial Buildings',
        'description': 'Synthetic hourly energy consumption data for US commercial building stock',
        'temporal_resolutions': ['15 minutes', '30 minutes', '60 minutes'],
        'date_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'reference': 'National Renewable Energy Laboratory (NREL) ComStock',
        'license': 'Open Data',
        'use_cases': [
            'End-use load disaggregation',
            'HVAC system detection',
            'Building energy analysis',
            'Appliance detection in commercial buildings'
        ]
    }
    
    # Analyze merged data files
    resolutions = ['15_Min', '30_Min', '60_Min']
    metadata['data_files'] = {}
    
    for resolution in resolutions:
        merged_file = data_path / f'comstock_merged_{resolution}.csv'
        label_file = data_path / f'comstock_{resolution.replace("_", "").lower()}_labels.csv'
        
        if merged_file.exists():
            print(f"Analyzing {merged_file.name}...")
            res_key = resolution.replace('_', ' ').lower()
            
            merged_stats = analyze_timeseries_data(merged_file)
            
            metadata['data_files'][res_key] = {
                'merged_data': {
                    'file': f'Comstock/comstock_merged_{resolution}.csv',
                    'description': f'Merged time series data at {res_key} resolution',
                    **merged_stats
                }
            }
            
            if label_file.exists():
                print(f"Analyzing {label_file.name}...")
                label_stats = analyze_timeseries_data(label_file)
                metadata['data_files'][res_key]['labels'] = {
                    'file': f'Comstock/comstock_{resolution.replace("_", "").lower()}_labels.csv',
                    'description': f'End-use load labels at {res_key} resolution',
                    **label_stats
                }
    
    # Analyze 15min detailed structure
    if comstock_15min_path.exists():
        metadata['comstock_15min_structure'] = {
            'base_path': 'Comstock_15min/',
            'description': 'Detailed 15-minute resolution data with separated inputs and labels'
        }
        
        # Analyze input file
        input_file = comstock_15min_path / 'Inputs' / 'x_comstock_1000_15min.csv'
        if input_file.exists():
            print(f"Analyzing {input_file}...")
            input_stats = analyze_timeseries_data(input_file)
            metadata['comstock_15min_structure']['input_data'] = {
                'file': 'Comstock_15min/Inputs/x_comstock_1000_15min.csv',
                'description': 'Total building energy consumption time series (1000 buildings)',
                **input_stats
            }
        
        # Analyze label files
        labels_dir = comstock_15min_path / 'Labels'
        if labels_dir.exists():
            label_files = list(labels_dir.glob('*.csv'))
            metadata['comstock_15min_structure']['end_use_categories'] = {
                'num_categories': len(label_files),
                'categories': []
            }
            
            for label_file in sorted(label_files):
                print(f"Analyzing {label_file.name}...")
                category_name = label_file.stem.replace('_case', '').replace('_', ' ').title()
                label_stats = analyze_labels(label_file)
                
                metadata['comstock_15min_structure']['end_use_categories']['categories'].append({
                    'name': category_name,
                    'file': f'Comstock_15min/Labels/{label_file.name}',
                    'description': f'Binary classification for {category_name} detection',
                    **label_stats
                })
    
    # Additional metadata
    metadata['building_characteristics'] = {
        'building_types': [
            'Office', 'Retail', 'School', 'Healthcare', 
            'Warehouse', 'Restaurant', 'Hotel', 'Others'
        ],
        'end_uses': [
            'Cooling', 'Heating', 'Interior Lighting', 'Exterior Lighting',
            'Interior Equipment', 'Fans', 'Pumps', 'Heat Rejection',
            'Heat Recovery', 'Water Systems', 'Refrigeration'
        ]
    }
    
    # Save metadata
    output_file = data_path / 'COMSTOCK_metadata.json'
    with open(output_file, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"\n✓ ComStock metadata saved to {output_file}")
    return metadata

def generate_restock_metadata():
    """Generate metadata for ResStock (US residential buildings) dataset"""
    
    data_path = root / 'data' / 'restock'
    
    if not data_path.exists():
        print("⚠ ResStock data directory not found, skipping...")
        return None
    
    metadata = {
        'dataset_name': 'ResStock - US Residential Building Stock',
        'dataset_type': 'Synthetic Building Energy Consumption Data',
        'source': 'NREL ResStock',
        'country': 'United States',
        'sector': 'Residential Buildings',
        'description': 'Synthetic hourly energy consumption data for US residential building stock',
        'temporal_resolutions': ['15 minutes', '30 minutes', '60 minutes'],
        'date_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'reference': 'National Renewable Energy Laboratory (NREL) ResStock',
        'license': 'Open Data'
    }
    
    # Analyze merged data files
    resolutions = ['15_Min', '30_Min', '60_Min']
    metadata['data_files'] = {}
    
    for resolution in resolutions:
        merged_file = data_path / f'restock_merged_{resolution}.csv'
        label_file = data_path / f'restock_{resolution.replace("_", "").lower()}_labels.csv'
        
        if merged_file.exists():
            print(f"Analyzing {merged_file.name}...")
            res_key = resolution.replace('_', ' ').lower()
            
            merged_stats = analyze_timeseries_data(merged_file)
            
            metadata['data_files'][res_key] = {
                'merged_data': {
                    'file': f'restock/restock_merged_{resolution}.csv',
                    'description': f'Merged time series data at {res_key} resolution',
                    **merged_stats
                }
            }
            
            if label_file.exists():
                label_stats = analyze_timeseries_data(label_file)
                metadata['data_files'][res_key]['labels'] = {
                    'file': f'restock/restock_{resolution.replace("_", "").lower()}_labels.csv',
                    'description': f'End-use load labels at {res_key} resolution',
                    **label_stats
                }
    
    # Save metadata
    output_file = data_path / 'RESTOCK_metadata.json'
    with open(output_file, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"\n✓ ResStock metadata saved to {output_file}")
    return metadata

def generate_summary_readme():
    """Generate a comprehensive README for the data directory"""
    
    readme_content = """# TransApp Dataset Documentation

## Overview
This directory contains datasets for training and evaluating TransApp models for appliance detection and energy disaggregation tasks.

## Datasets

### 1. CER (Commission for Energy Regulation) - Irish Residential
- **Location**: `data/Inputs/`, `data/Labels/`, `data/ExogeneData/`
- **Type**: Real smart meter data from residential customers in Ireland
- **Resolution**: 30 minutes
- **Use Case**: Residential appliance detection
- **Metadata**: See `CER_metadata.json`

**Structure**:
```
Inputs/
  x_residential_25728.csv     # Consumption time series (25,728 samples)
Labels/
  cooker_case.csv             # Binary labels for cooker detection
  dishwasher_case.csv         # Binary labels for dishwasher detection
  waterheater_case.csv        # Binary labels for water heater detection
  ... (9 appliance categories total)
ExogeneData/
  extra_25728.csv             # Temporal features (hours, days in sin/cos)
```

### 2. ComStock - US Commercial Buildings
- **Location**: `data/Comstock/`, `data/Comstock_15min/`
- **Type**: Synthetic energy data for commercial buildings (NREL)
- **Resolutions**: 15 min, 30 min, 60 min
- **Use Case**: Commercial building end-use disaggregation
- **Metadata**: See `Comstock/COMSTOCK_metadata.json`

**Structure**:
```
Comstock/
  comstock_merged_15_Min.csv   # Merged data at 15-min resolution
  comstock_15min_labels.csv    # End-use labels at 15-min resolution
  comstock_merged_30_Min.csv   # Merged data at 30-min resolution
  comstock_30min_labels.csv    # End-use labels at 30-min resolution
  comstock_merged_60_Min.csv   # Merged data at 60-min resolution
  comstock_60min_labels.csv    # End-use labels at 60-min resolution

Comstock_15min/
  Inputs/
    x_comstock_1000_15min.csv  # Total building consumption (1000 buildings)
  Labels/
    cooling_case.csv           # Binary labels for cooling detection
    heating_case.csv           # Binary labels for heating detection
    interior_lighting_case.csv # Binary labels for interior lighting
    ... (11 end-use categories total)
```

**End-Use Categories**:
- Cooling
- Heating
- Interior Lighting
- Exterior Lighting
- Interior Equipment
- Fans
- Pumps
- Heat Rejection
- Heat Recovery
- Water Systems
- Refrigeration

### 3. ResStock - US Residential Buildings
- **Location**: `data/restock/`
- **Type**: Synthetic energy data for residential buildings (NREL)
- **Resolutions**: 15 min, 30 min, 60 min
- **Use Case**: Residential building end-use disaggregation
- **Metadata**: See `restock/RESTOCK_metadata.json`

## Data Loading Functions

Use the provided utility functions in `experiments/data_utils.py`:

### CER Dataset
```python
# Pretraining data
data = CER_get_data_pretraining(
    exo_variable=['hours_cos', 'hours_sin', 'days_cos', 'days_sin']
)

# Classification data
train_x, train_y, valid_x, valid_y, test_x, test_y, \
train_voter_x, train_voter_y, valid_voter_x, valid_voter_y, \
test_voter_x, test_voter_y = CER_get_data_case(
    'cooker_case', seed=0, win=128
)
```

### ComStock Dataset
```python
# Pretraining data
data = COMSTOCK_get_data_pretraining(resolution='15min')

# Classification data
datas_tuple = COMSTOCK_get_data_case('cooling_case', seed=0)
```

## File Formats

All CSV files follow these conventions:
- **Input files**: Columns represent time series features, rows represent time steps
- **Label files**: Binary classification labels (0/1) for presence/absence of appliance
- **Exogenous files**: Temporal features encoded as sin/cos transformations

## Citation

If you use these datasets, please cite:

**CER Dataset**:
```
Commission for Energy Regulation (CER)
Smart Metering Project - Electricity Customer Behaviour Trial
```

**ComStock/ResStock Datasets**:
```
National Renewable Energy Laboratory (NREL)
End-Use Load Profiles for the U.S. Building Stock
https://www.nrel.gov/buildings/end-use-load-profiles.html
```

**TransApp Framework**:
```
@article{petralia2023transapp,
  title={TransApp: A Transformer-Based Framework for Appliance Detection Using Smart Meter Consumption Series},
  author={Petralia, Adrien and others},
  journal={EDF R&D and Université Paris Cité},
  year={2023}
}
```

## Preprocessing

Raw data has been preprocessed with:
- Normalization/standardization
- Missing value handling
- Temporal resampling (where applicable)
- Train/validation/test splits

See `data/preprocess.ipynb` for preprocessing details.

## License

- **CER Dataset**: Open Data
- **ComStock/ResStock**: NREL Open Data
- **TransApp Code**: ©2023 EDF

## Contact

For questions or issues with the data, please refer to the original dataset sources or contact the TransApp development team.

---
*Last updated: {}*
""".format(datetime.now().strftime('%Y-%m-%d'))
    
    output_file = root / 'data' / 'README_DATASETS.md'
    with open(output_file, 'w') as f:
        f.write(readme_content)
    
    print(f"\n✓ Summary README saved to {output_file}")

def main():
    print("=" * 70)
    print("         TransApp Dataset Metadata Generation")
    print("=" * 70)
    print()
    
    # Generate CER metadata
    print("📊 Generating CER metadata...")
    print("-" * 70)
    cer_meta = generate_cer_metadata()
    print()
    
    # Generate ComStock metadata
    print("📊 Generating ComStock metadata...")
    print("-" * 70)
    comstock_meta = generate_comstock_metadata()
    print()
    
    # Generate ResStock metadata
    print("📊 Generating ResStock metadata...")
    print("-" * 70)
    restock_meta = generate_restock_metadata()
    print()
    
    # Generate summary README
    print("📝 Generating summary README...")
    print("-" * 70)
    generate_summary_readme()
    print()
    
    print("=" * 70)
    print("✓ All metadata files generated successfully!")
    print("=" * 70)
    print()
    print("Generated files:")
    print("  - data/CER_metadata.json")
    print("  - data/Comstock/COMSTOCK_metadata.json")
    if restock_meta:
        print("  - data/restock/RESTOCK_metadata.json")
    print("  - data/README_DATASETS.md")
    print()

if __name__ == "__main__":
    main()

         TransApp Dataset Metadata Generation

📊 Generating CER metadata...
----------------------------------------------------------------------
Analyzing /home/vanshdhar/Desktop/ISP/TransApp/data/Inputs/x_residential_25728.csv...
Analyzing /home/vanshdhar/Desktop/ISP/TransApp/data/ExogeneData/extra_25728.csv...
Analyzing cooker_case.csv...
Analyzing desktopcomputer_case.csv...
Analyzing dishwasher_case.csv...
Analyzing laptopcomputer_case.csv...
Analyzing pluginheater_case.csv...
Analyzing tumbledryer_case.csv...
Analyzing tv_greater21inch_case.csv...
Analyzing tv_less21inch_case.csv...
Analyzing waterheater_case.csv...

✓ CER metadata saved to /home/vanshdhar/Desktop/ISP/TransApp/data/CER_metadata.json

📊 Generating ComStock metadata...
----------------------------------------------------------------------
Analyzing comstock_merged_15_Min.csv...
Analyzing comstock_15min_labels.csv...
Analyzing comstock_merged_30_Min.csv...
Analyzing comstock_30min_labels.csv...
Analyzing comstock_m

ValueError: unexpected '{' in field name